In [2]:
import json
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")

# Verificar a quantidade de jogos puxados
with open(DATA_DIR / "steamspy_raw.json", encoding="utf-8") as f:
    raw = json.load(f)

print(f"Total de jogos: {len(raw)}")

Total de jogos: 4996


In [3]:
# pega o primeiro item do dict pra inspecionar a estrutura
primeiro_appid = list(raw.keys())[0]
raw[primeiro_appid]

{'appid': 730,
 'name': 'Counter-Strike: Global Offensive',
 'developer': 'Valve',
 'publisher': 'Valve',
 'score_rank': '',
 'positive': 7642084,
 'negative': 1173003,
 'userscore': 0,
 'owners': '100,000,000 .. 200,000,000',
 'average_forever': 0,
 'average_2weeks': 0,
 'median_forever': 0,
 'median_2weeks': 0,
 'price': '0',
 'initialprice': '0',
 'discount': '0',
 'ccu': 1013936}

In [4]:
# Transforma em tabela
df = pd.DataFrame(raw.values())
df.head()

,appid,name,developer,publisher,score_rank,positive,negative,userscore,owners,average_forever,average_2weeks,median_forever,median_2weeks,price,initialprice,discount,ccu
0,730,Counter-Strike: Global Offensive,Valve,Valve,,7642084,1173003,0,"100,000,000 .. 200,000,000",0,0,0,0,0,0,0,1013936
1,1172470,Apex Legends,Respawn,Electronic Arts,,668053,326926,0,"100,000,000 .. 200,000,000",0,0,0,0,0,0,0,124262
2,578080,PUBG: BATTLEGROUNDS,PUBG Corporation,"KRAFTON, Inc.",,1520457,1037487,0,"100,000,000 .. 200,000,000",0,0,0,0,0,0,0,314682
3,1623730,Palworld,Pocketpair,Pocketpair,,358266,22443,0,"50,000,000 .. 100,000,000",0,0,0,0,2999,2999,0,18028
4,440,Team Fortress 2,Valve,Valve,,1044264,117208,0,"50,000,000 .. 100,000,000",0,0,0,0,0,0,0,43819


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4996 entries, 0 to 4995
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   appid            4996 non-null   int64 
 1   name             4996 non-null   str   
 2   developer        4996 non-null   str   
 3   publisher        4996 non-null   str   
 4   score_rank       4996 non-null   object
 5   positive         4996 non-null   int64 
 6   negative         4996 non-null   int64 
 7   userscore        4996 non-null   int64 
 8   owners           4996 non-null   str   
 9   average_forever  4996 non-null   int64 
 10  average_2weeks   4996 non-null   int64 
 11  median_forever   4996 non-null   int64 
 12  median_2weeks    4996 non-null   int64 
 13  price            4996 non-null   str   
 14  initialprice     4996 non-null   str   
 15  discount         4996 non-null   str   
 16  ccu              4996 non-null   int64 
dtypes: int64(9), object(1), str(7)
memory usage:

In [6]:
# Normalizar a quantidade de proprietários
def parse_owners_range(owners_str):
    partes = owners_str.replace(",", "").split("..")
    baixo, alto = (int(p.strip()) for p in partes)
    return (baixo + alto) / 2

df["owners_estimate"] = df["owners"].apply(parse_owners_range)
df[["name", "owners", "owners_estimate"]].head()

,name,owners,owners_estimate
0,Counter-Strike: Global Offensive,"100,000,000 .. 200,000,000",150000000.0
1,Apex Legends,"100,000,000 .. 200,000,000",150000000.0
2,PUBG: BATTLEGROUNDS,"100,000,000 .. 200,000,000",150000000.0
3,Palworld,"50,000,000 .. 100,000,000",75000000.0
4,Team Fortress 2,"50,000,000 .. 100,000,000",75000000.0


In [7]:
# Passando colunas de str para int
df["price"] = pd.to_numeric(df["price"], errors = "coerce") / 100
df["initialprice"] = pd.to_numeric(df["initialprice"], errors = "coerce") / 100
df["discount"] = pd.to_numeric(df["discount"], errors = "coerce")

df[["name", "price", "initialprice", "discount"]].head()

,name,price,initialprice,discount
0,Counter-Strike: Global Offensive,0.00,0.00,0
1,Apex Legends,0.00,0.00,0
2,PUBG: BATTLEGROUNDS,0.00,0.00,0
3,Palworld,29.99,29.99,0
4,Team Fortress 2,0.00,0.00,0


In [8]:
# Criar novas colunas para cálculo de reviews
df["total_reviews"] = df["positive"] + df["negative"]
df["positive_pct"] = (df["positive"] / df["total_reviews"].replace(0, pd.NA) * 100)
df["positive_pct"] = pd.to_numeric(df["positive_pct"], errors = "coerce").round(2)

df[["name", "positive", "negative", "total_reviews", "positive_pct"]].head()

,name,positive,negative,total_reviews,positive_pct
0,Counter-Strike: Global Offensive,7642084,1173003,8815087,86.69
1,Apex Legends,668053,326926,994979,67.14
2,PUBG: BATTLEGROUNDS,1520457,1037487,2557944,59.44
3,Palworld,358266,22443,380709,94.10
4,Team Fortress 2,1044264,117208,1161472,89.91


In [9]:
# Verificar colunas vazias
print("score_rank vazio:", (df["score_rank"] == "").sum(), "de", len(df))
print("average_forever == 0:", (df["average_forever"] == 0).sum(), "de", len(df))
print("average_2weeks == 0:", (df["average_2weeks"] == 0).sum(), "de", len(df))
print("median_forever == 0:", (df["median_forever"] == 0).sum(), "de", len(df))
print("median_2weeks == 0:", (df["median_2weeks"] == 0).sum(), "de", len(df))
print("userscore == 0:", (df["userscore"] == 0).sum(), "de", len(df))


score_rank vazio: 4995 de 4996
average_forever == 0: 4996 de 4996
average_2weeks == 0: 4996 de 4996
median_forever == 0: 4996 de 4996
median_2weeks == 0: 4996 de 4996
userscore == 0: 4995 de 4996


In [10]:
# Deletar colunas vazias
df = df.drop(columns=["score_rank", "average_forever", "average_2weeks", "median_forever", "median_2weeks", "owners", "userscore"])
df.head()

,appid,name,developer,publisher,positive,negative,price,initialprice,discount,ccu,owners_estimate,total_reviews,positive_pct
0,730,Counter-Strike: Global Offensive,Valve,Valve,7642084,1173003,0.00,0.00,0,1013936,150000000.0,8815087,86.69
1,1172470,Apex Legends,Respawn,Electronic Arts,668053,326926,0.00,0.00,0,124262,150000000.0,994979,67.14
2,578080,PUBG: BATTLEGROUNDS,PUBG Corporation,"KRAFTON, Inc.",1520457,1037487,0.00,0.00,0,314682,150000000.0,2557944,59.44
3,1623730,Palworld,Pocketpair,Pocketpair,358266,22443,29.99,29.99,0,18028,75000000.0,380709,94.10
4,440,Team Fortress 2,Valve,Valve,1044264,117208,0.00,0.00,0,43819,75000000.0,1161472,89.91


In [11]:
# Remover duplicados
print("Duplicados:", df.duplicated(subset="appid").sum())
df = df.drop_duplicates(subset="appid")

Duplicados: 0


In [12]:
# Jogos sem owners
print("Sem owners válido:", df["owners_estimate"].isna().sum())
df = df[df["owners_estimate"].notna()]

Sem owners válido: 0


In [ ]:
# Criando o arquivo CSV ja limpo
df.to_csv("../data/steam_games_clean.csv", index=False)
print(f"{len(df)} jogos salvos.")

4996 jogos salvos.
